<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

##Signals I will check

I will first check two signals before defining the baseline rule. The first is content staleness, measured using days_since_last_update, because older pages may be candidates for refresh. The second is CTR in relation to average search position, because a page that receives impressions but has relatively weak CTR may deserve review. I will only use these signals in the final rule if the observed data supports them.

In [8]:
import os
print(os.getcwd())
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

/content/flyrank-ml-internship
True


In [9]:
import os
import subprocess

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print(
    "Dataset found:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Working directory: /content/flyrank-ml-internship
Dataset found: True


In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

stale_bins = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, float("inf")],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

stale_table = (
    df.assign(staleness_bucket=stale_bins)
      .groupby("staleness_bucket", observed=False)
      .agg(
          pages=("content_id", "count"),
          declining_rate=("trend_direction", lambda s: (s == "down").mean())
      )
)

print(stale_table)

                  pages  declining_rate
staleness_bucket                       
0-30              20480        0.511377
31-90               175        0.588571
91-180             9171        0.611057
181-365             169        0.467456
365+                  5        0.600000


##Signal 1 — Staleness: MIXED

The observed data shows some evidence that older pages may be more likely to decline, especially in the 91–180 day bucket, where the declining rate is about 61.1% compared with about 51.1% for pages updated within 30 days. However, the pattern is not consistent across all buckets, and some older buckets contain very few pages. Therefore, I would treat staleness as a useful but imperfect signal rather than a rule by itself.

In [11]:
position_bins = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

ctr_table = (
    df.assign(position_bucket=position_bins)
      .groupby("position_bucket", observed=False)
      .agg(
          pages=("content_id", "count"),
          median_ctr=("ctr", "median"),
          declining_rate=("trend_direction", lambda s: (s == "down").mean())
      )
)

print(ctr_table)

                 pages  median_ctr  declining_rate
position_bucket                                   
1-3               1141        0.00        0.497809
4-10             11842        0.16        0.569414
11-20             7273        0.10        0.609515
21-50             7225        0.03        0.561799
50+               1314        0.00        0.343227


##Signal 2 — CTR relative to position: MIXED

The observed data suggests that CTR and search position together contain useful information, but the relationship is not simple or monotonic. Pages in positions 11–20 have the highest declining rate at about 61.0%, while pages beyond position 50 have a much lower declining rate of about 34.3%. This means CTR and position may help identify pages for review, but they should not be used as a single rigid rule on their own.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

##Building the ranked queue

I will convert the two observed signals into a simple baseline score. A page receives one point if it is stale and one point if it has low CTR for its search position. Pages with both signals will be reviewed first, pages with one signal will be reviewed next, and pages with neither signal will have low priority.

The score is intentionally simple so that every recommendation can be explained. I will rank pages by the baseline score and use impressions as a secondary sorting signal so that higher-visibility pages are reviewed first when scores are tied.

In [12]:
import os
import numpy as np

# Make sure output folder exists
os.makedirs("work/outputs", exist_ok=True)

work = df.copy()

# Baseline conditions
work["is_stale"] = work["days_since_last_update"] >= 90

work["low_ctr_for_position"] = (
    ((work["avg_position"] >= 4) & (work["avg_position"] <= 20) & (work["ctr"] < 0.10))
)

# Simple interpretable score
work["baseline_score"] = (
    work["is_stale"].astype(int) * 1
    + work["low_ctr_for_position"].astype(int) * 1
)

# Reason code
work["reason_code"] = np.select(
    [
        work["is_stale"] & work["low_ctr_for_position"],
        work["is_stale"],
        work["low_ctr_for_position"]
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE_PAGE",
        "LOW_CTR_FOR_POSITION"
    ],
    default="LOW_PRIORITY"
)

# Action label
work["action"] = np.select(
    [
        work["baseline_score"] == 2,
        work["baseline_score"] == 1
    ],
    [
        "REVIEW_FIRST",
        "REVIEW"
    ],
    default="LOW_PRIORITY"
)

# Rank pages
ranked = work.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

# Save required CSV
csv_path = "work/outputs/baseline_action_score.csv"

ranked[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].to_csv(csv_path, index=False)

print("CSV written to:", csv_path)
print("Rows:", len(ranked))

ranked[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].head(10)

CSV written to: work/outputs/baseline_action_score.csv
Rows: 30000


,content_id,baseline_score,reason_code,action,impressions_90d,ctr,avg_position,days_since_last_update
3394,content_36ff89c8214e,2,STALE_AND_LOW_CTR,REVIEW_FIRST,295097,0.05,7.3,104
7445,content_c8e9d6ab9013,2,STALE_AND_LOW_CTR,REVIEW_FIRST,208678,0.00,9.7,104
3070,content_91652435f57a,2,STALE_AND_LOW_CTR,REVIEW_FIRST,159590,0.06,7.8,104
5621,content_97a86caf3a3d,2,STALE_AND_LOW_CTR,REVIEW_FIRST,147670,0.07,6.4,104
9193,content_c1fe78bc4e37,2,STALE_AND_LOW_CTR,REVIEW_FIRST,134055,0.03,7.5,104
2476,content_4c76e9b13aea,2,STALE_AND_LOW_CTR,REVIEW_FIRST,127952,0.07,7.4,104
4708,content_b115f7c74779,2,STALE_AND_LOW_CTR,REVIEW_FIRST,123469,0.03,8.0,104
1034,content_647e177596e9,2,STALE_AND_LOW_CTR,REVIEW_FIRST,111222,0.09,7.4,104
22066,content_42d423551e2c,2,STALE_AND_LOW_CTR,REVIEW_FIRST,106652,0.09,4.8,104
18216,content_5d3dfb80a423,2,STALE_AND_LOW_CTR,REVIEW_FIRST,97235,0.07,6.5,104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

##Top-ranked page review

I reviewed the highest-ranked pages produced by the baseline rule. These pages appear at the top because they satisfy both conditions: they have not been updated recently and they also show relatively low CTR for their search position. I treat these results as decision-support rather than proof that the pages are truly declining.

For each result, the suggested action is to review the page first. The reason code is STALE_AND_LOW_CTR. My confidence is moderate because the rule is based on two observed signals, but both signals showed mixed relationships in the earlier checks.

A recommendation could be wrong if the page is intentionally old, if low CTR is normal for its query or content type, if seasonality explains the pattern, or if the page is still performing well for business reasons not represented in this dataset.

In [13]:
review_cols = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

top20_review = ranked[review_cols].head(20).copy()

top20_review["confidence_note"] = "Moderate"
top20_review["what_would_make_it_wrong"] = (
    "Old content may still be appropriate, low CTR may be normal for the query, "
    "or important context may be missing from the dataset."
)

top20_review



,content_id,baseline_score,reason_code,action,impressions_90d,ctr,avg_position,days_since_last_update,confidence_note,what_would_make_it_wrong
3394,content_36ff89c8214e,2,STALE_AND_LOW_CTR,REVIEW_FIRST,295097,0.05,7.3,104,Moderate,"Old content may still be appropriate, low CTR ..."
7445,content_c8e9d6ab9013,2,STALE_AND_LOW_CTR,REVIEW_FIRST,208678,0.00,9.7,104,Moderate,"Old content may still be appropriate, low CTR ..."
3070,content_91652435f57a,2,STALE_AND_LOW_CTR,REVIEW_FIRST,159590,0.06,7.8,104,Moderate,"Old content may still be appropriate, low CTR ..."
5621,content_97a86caf3a3d,2,STALE_AND_LOW_CTR,REVIEW_FIRST,147670,0.07,6.4,104,Moderate,"Old content may still be appropriate, low CTR ..."
9193,content_c1fe78bc4e37,2,STALE_AND_LOW_CTR,REVIEW_FIRST,134055,0.03,7.5,104,Moderate,"Old content may still be appropriate, low CTR ..."
2476,content_4c76e9b13aea,2,STALE_AND_LOW_CTR,REVIEW_FIRST,127952,0.07,7.4,104,Moderate,"Old content may still be appropriate, low CTR ..."
4708,content_b115f7c74779,2,STALE_AND_LOW_CTR,REVIEW_FIRST,123469,0.03,8.0,104,Moderate,"Old content may still be appropriate, low CTR ..."
1034,content_647e177596e9,2,STALE_AND_LOW_CTR,REVIEW_FIRST,111222,0.09,7.4,104,Moderate,"Old content may still be appropriate, low CTR ..."
22066,content_42d423551e2c,2,STALE_AND_LOW_CTR,REVIEW_FIRST,106652,0.09,4.8,104,Moderate,"Old content may still be appropriate, low CTR ..."
18216,content_5d3dfb80a423,2,STALE_AND_LOW_CTR,REVIEW_FIRST,97235,0.07,6.5,104,Moderate,"Old content may still be appropriate, low CTR ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##Weak picks and leakage check

Some high-ranked pages may still be weak recommendations because the baseline rule is intentionally simple. A page can be stale and have low CTR without actually needing intervention. For example, the page may be seasonal, intentionally unchanged, or serving a query where low CTR is normal.

I also checked that the baseline score does not use the outcome variable trend_direction, future-window information, or any product decision flag. The ranking only uses observable page features available before the review decision, so the rule is designed to avoid obvious target leakage.

In [14]:
# Show a few weaker examples from the top-ranked set
weak_picks = top20_review.tail(5).copy()

print("Example weaker picks from the top-ranked set:")
display(
    weak_picks[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ]
)

# Leakage check
features_used = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

forbidden_inputs = [
    "trend_direction"
]

print("\nFeatures used in ranking:", features_used)
print("Forbidden outcome used:", any(x in features_used for x in forbidden_inputs))


Example weaker picks from the top-ranked set:


,content_id,baseline_score,reason_code,action,impressions_90d,ctr,avg_position,days_since_last_update
29775,content_896bf2cc27b7,2,STALE_AND_LOW_CTR,REVIEW_FIRST,66359,0.04,4.9,104
4895,content_d07ea098353c,2,STALE_AND_LOW_CTR,REVIEW_FIRST,63366,0.03,9.4,104
5933,content_00202ac57009,2,STALE_AND_LOW_CTR,REVIEW_FIRST,61832,0.09,18.0,104
24346,content_156df458c02c,2,STALE_AND_LOW_CTR,REVIEW_FIRST,55281,0.09,6.5,104
9308,content_9c8299b55f3c,2,STALE_AND_LOW_CTR,REVIEW_FIRST,54783,0.03,8.5,104



Features used in ranking: ['days_since_last_update', 'ctr', 'avg_position', 'impressions_90d']
Forbidden outcome used: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.